#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [2]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [3]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

In [4]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [5]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [7]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [8]:
train_data = pd.read_csv('data01_train.csv')
test_data = pd.read_csv('data01_test.csv')

#### 2) 기본 정보 조회

In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [10]:
train_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [11]:
train_data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


In [12]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1471 entries, 0 to 1470
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 6.3+ MB


In [13]:
test_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.284379,-0.021981,-0.116683,-0.992490,-0.979640,-0.963321,-0.992563,-0.977304,-0.958142,-0.938850,...,-0.850065,-0.018043,0.092304,0.074220,-0.714534,-0.671943,-0.018351,-0.185733,22,SITTING
1,0.277440,-0.028086,-0.118412,-0.996620,-0.927676,-0.972294,-0.997346,-0.931405,-0.971788,-0.939837,...,-0.613367,-0.022456,-0.155414,0.247498,-0.112257,-0.826816,0.184489,-0.068699,15,STANDING
2,0.305833,-0.041023,-0.087303,0.006880,0.182800,-0.237984,0.005642,0.028616,-0.236474,0.016311,...,0.394388,-0.362616,0.171069,0.576349,-0.688314,-0.743234,0.272186,0.053101,22,WALKING
3,0.276053,-0.016487,-0.108381,-0.995379,-0.983978,-0.975854,-0.995877,-0.985280,-0.974907,-0.941425,...,-0.841455,0.289548,0.079801,-0.020033,0.291898,-0.639435,-0.111998,-0.123298,8,SITTING
4,0.271998,0.016904,-0.078856,-0.973468,-0.702462,-0.869450,-0.979810,-0.711601,-0.856807,-0.920760,...,0.214219,0.010111,0.114179,-0.830776,-0.325098,-0.840817,0.116237,-0.096615,5,STANDING


In [14]:
test_data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,...,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000,1471.000000
mean,0.273198,-0.017281,-0.108123,-0.614634,-0.515427,-0.607533,-0.639950,-0.530760,-0.609256,-0.481055,...,-0.311513,-0.632273,0.009324,0.015666,0.006272,-0.001510,-0.481737,0.055771,-0.064194,17.539089
std,0.079989,0.045957,0.049082,0.448480,0.506094,0.424243,0.422994,0.489381,0.418536,0.542756,...,0.313792,0.296179,0.324864,0.452616,0.608954,0.483028,0.522714,0.298124,0.281645,9.122876
min,-1.000000,-1.000000,-0.418354,-0.999717,-0.999873,-1.000000,-0.999867,-1.000000,-0.999879,-0.948723,...,-0.995357,-0.994664,-0.937468,-0.990492,-0.995222,-0.969066,-0.999380,-0.995073,-1.000000,1.000000
25%,0.263787,-0.024792,-0.120733,-0.992669,-0.979082,-0.981070,-0.993498,-0.979214,-0.980659,-0.936791,...,-0.546342,-0.844547,-0.119166,-0.265533,-0.485998,-0.380300,-0.814060,-0.017413,-0.148445,8.000000
50%,0.277322,-0.017187,-0.108124,-0.952426,-0.867309,-0.869890,-0.958705,-0.873891,-0.863451,-0.890491,...,-0.347433,-0.706699,0.005049,0.023421,-0.005036,0.002408,-0.708911,0.178814,-0.002243,19.000000
75%,0.288058,-0.010238,-0.096606,-0.245405,-0.030639,-0.260223,-0.297260,-0.058824,-0.256657,-0.030692,...,-0.125796,-0.516910,0.135698,0.312690,0.518184,0.374583,-0.486534,0.248126,0.096674,26.000000
max,0.631510,0.359587,0.543939,0.899922,0.782590,0.931308,0.950758,0.602458,0.784041,0.821218,...,0.941113,0.894210,0.980889,0.991899,0.994366,0.979522,1.000000,0.432496,0.992766,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [15]:
train_data.drop('subject', axis=1, inplace=True)
test_data.drop('subject', axis=1, inplace=True)

In [16]:
sum(train_data.isna().sum())

0

In [17]:
sum(test_data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [18]:
dynamic_activity = (train_data['Activity']=='WALKING') | (train_data['Activity']=='WALKING_UPSTAIRS') | (train_data['Activity']=='WALKING_DOWNSTAIRS')
train_data = train_data[dynamic_activity]

In [19]:
train_data

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,WALKING_DOWNSTAIRS
5,0.330708,0.007561,-0.061371,-0.215760,0.101075,0.072949,-0.269857,0.060060,0.101298,-0.019263,...,-0.599906,-0.887024,-0.030645,-0.852091,-0.500195,0.306091,-0.552729,0.253885,0.291256,WALKING_UPSTAIRS
6,0.121465,-0.031902,-0.005196,-0.152198,-0.113104,-0.239423,-0.202401,-0.164698,-0.247099,0.114668,...,-0.371055,-0.775779,0.445206,-0.003487,-0.940185,0.041387,-0.886603,0.173338,-0.005627,WALKING
12,0.303885,0.002768,-0.038613,-0.168656,0.190336,-0.140473,-0.205134,0.101144,-0.120572,-0.000818,...,-0.037750,-0.329728,-0.040030,0.257252,0.076091,-0.123425,-0.752882,0.266729,0.045692,WALKING
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5868,0.277709,-0.053919,-0.098746,-0.589970,-0.076626,-0.500837,-0.605474,-0.142798,-0.506696,-0.533485,...,0.029201,-0.268237,-0.076922,0.706620,-0.954969,-0.324779,-0.691663,0.153974,-0.196833,WALKING
5869,0.252496,-0.009773,-0.118293,-0.155168,-0.244513,-0.044524,-0.244367,-0.229509,-0.042199,0.291682,...,-0.475800,-0.752980,0.619246,-0.684483,0.924623,-0.353041,-0.775518,0.253218,0.013216,WALKING_UPSTAIRS
5873,0.264961,-0.034719,-0.088140,-0.368158,-0.074514,-0.229606,-0.394387,-0.082437,-0.243525,-0.261087,...,0.198624,-0.046157,0.037517,-0.903692,0.614933,-0.610918,-0.601604,0.335996,0.156518,WALKING
5877,0.191568,0.013328,-0.105174,-0.126969,-0.121729,-0.327480,-0.192523,-0.109923,-0.295286,0.078644,...,-0.622617,-0.879215,0.721718,0.623151,0.866858,-0.445660,-0.690278,0.303194,-0.044188,WALKING_UPSTAIRS


In [20]:
dynamic_activity = (test_data['Activity']=='WALKING') | (test_data['Activity']=='WALKING_UPSTAIRS') | (test_data['Activity']=='WALKING_DOWNSTAIRS')
test_data = test_data[dynamic_activity]

In [21]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [22]:
X_test = test_data.drop('Activity', axis=1)
y_test = test_data.loc[:, 'Activity']

In [23]:
y_train.unique()

array(['WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'], dtype=object)

In [24]:
y_test.unique()

array(['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS'], dtype=object)

In [25]:
X_train

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,0.050888,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.013902,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758
5,0.330708,0.007561,-0.061371,-0.215760,0.101075,0.072949,-0.269857,0.060060,0.101298,-0.019263,...,0.114524,-0.599906,-0.887024,-0.030645,-0.852091,-0.500195,0.306091,-0.552729,0.253885,0.291256
6,0.121465,-0.031902,-0.005196,-0.152198,-0.113104,-0.239423,-0.202401,-0.164698,-0.247099,0.114668,...,-0.127764,-0.371055,-0.775779,0.445206,-0.003487,-0.940185,0.041387,-0.886603,0.173338,-0.005627
12,0.303885,0.002768,-0.038613,-0.168656,0.190336,-0.140473,-0.205134,0.101144,-0.120572,-0.000818,...,0.227840,-0.037750,-0.329728,-0.040030,0.257252,0.076091,-0.123425,-0.752882,0.266729,0.045692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5868,0.277709,-0.053919,-0.098746,-0.589970,-0.076626,-0.500837,-0.605474,-0.142798,-0.506696,-0.533485,...,-0.058903,0.029201,-0.268237,-0.076922,0.706620,-0.954969,-0.324779,-0.691663,0.153974,-0.196833
5869,0.252496,-0.009773,-0.118293,-0.155168,-0.244513,-0.044524,-0.244367,-0.229509,-0.042199,0.291682,...,0.263950,-0.475800,-0.752980,0.619246,-0.684483,0.924623,-0.353041,-0.775518,0.253218,0.013216
5873,0.264961,-0.034719,-0.088140,-0.368158,-0.074514,-0.229606,-0.394387,-0.082437,-0.243525,-0.261087,...,0.155018,0.198624,-0.046157,0.037517,-0.903692,0.614933,-0.610918,-0.601604,0.335996,0.156518
5877,0.191568,0.013328,-0.105174,-0.126969,-0.121729,-0.327480,-0.192523,-0.109923,-0.295286,0.078644,...,0.325461,-0.622617,-0.879215,0.721718,0.623151,0.866858,-0.445660,-0.690278,0.303194,-0.044188


### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [26]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [27]:
# 훈련 데이터 전체로 교차 검증할 예정
# x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [28]:
kappa_df = pd.DataFrame({'모델': [],
                       'gs_kappa(교차검증)': [],
                       'gs_kappa(테스트)': [],
                      })
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트)


In [29]:
kappa_scorer = make_scorer(cohen_kappa_score)

### (1) 알고리즘1 : RandomForest

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [ ]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 120}
0.9800758851536248


In [ ]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.987460815047022
[[225   0   3]
 [  3 191   1]
 [  0   1 214]]
                    precision    recall  f1-score   support

           WALKING       0.99      0.99      0.99       228
WALKING_DOWNSTAIRS       0.99      0.98      0.99       195
  WALKING_UPSTAIRS       0.98      1.00      0.99       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9811


In [ ]:
joblib.dump(rf_model, 'rf_model_gs.pkl')

['rf_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'RandomForest', 'gs_kappa(교차검증)': gs.best_score_, 'gs_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트)
0,RandomForest,0.980076,0.981149


### (2) 알고리즘2 : LogisticRegressor

In [30]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [31]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}],
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [32]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 100, 'penalty': 'l2', 'solver': 'liblinear'}
0.9977215093314008


In [33]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [34]:
joblib.dump(lr_model, 'lr_model_gs.pkl')

['lr_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', 'gs_kappa(교차검증)': gs.best_score_, 'gs_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트)
0,RandomForest,0.980076,0.981149
1,LogisticRegression,0.997722,0.997644


### (3) 알고리즘3 : SVM

In [ ]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [ ]:
gs = GridSearchCV(SVC(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 1, 'kernel': 'linear'}
0.9977206043335023


In [ ]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9952978056426333
[[227   0   1]
 [  0 194   1]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       0.99      1.00      0.99       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9929


In [ ]:
joblib.dump(svm_model, 'svm_model_gs.pkl')

['svm_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'SVM', 'gs_kappa(교차검증)': gs.best_score_, 'gs_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트)
0,RandomForest,0.980076,0.981149
1,LogisticRegression,0.997722,0.997644
2,SVM,0.997721,0.992932


### (4) 알고리즘4 : DecisionTreeClassifier

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8}
0.9048574359613297


In [ ]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9310344827586207
[[211   6  11]
 [  1 187   7]
 [  8  11 196]]
                    precision    recall  f1-score   support

           WALKING       0.96      0.93      0.94       228
WALKING_DOWNSTAIRS       0.92      0.96      0.94       195
  WALKING_UPSTAIRS       0.92      0.91      0.91       215

          accuracy                           0.93       638
         macro avg       0.93      0.93      0.93       638
      weighted avg       0.93      0.93      0.93       638

Cohen's Kappa Score: 0.8964


In [ ]:
joblib.dump(dt_model, 'dt_model_gs.pkl')

['dt_model_gs.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', 'gs_kappa(교차검증)': gs.best_score_, 'gs_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트)
0,RandomForest,0.980076,0.981149
1,LogisticRegression,0.997722,0.997644
2,SVM,0.997721,0.992932
3,DecisionTreeClassifier,0.904857,0.896449


In [ ]:
kappa_df.to_csv('모델 kappa(그리드서치).csv', index=False, encoding='utf-8-sig')

### (5) 알고리즘5 : XGBoost

In [ ]:
# # XGBoost는 label로 숫자를 원함.
# le = LabelEncoder()
# y_xgb_train = le.fit_transform(y_train)

In [ ]:
# XGB_model = XGBClassifier(random_state=42)
# scores = cross_validate(XGB_model, X_train, y_xgb_train, cv=5)
# print(np.mean(scores['test_score']))

In [ ]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train, y_xgb_train)
# y_pred = XGB_model.predict(X_test)
# y_pred = le.inverse_transform(y_pred)

# print('accuracy :',accuracy_score(y_test, y_pred))
# print('='*60)
# print(confusion_matrix(y_test, y_pred))
# print('='*60)
# print(classification_report(y_test, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_test, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

### (6) 알고리즘6 : LightGBM

In [ ]:
# X_train_lgb = X_train.copy()
# X_train_lgb.columns = [f'feature_{i}' for i in range(X_train.shape[1])]

In [ ]:
# X_test_lgb = X_test.copy()
# X_test_lgb.columns = [f'feature_{i}' for i in range(X_test.shape[1])]

In [ ]:
# le = LabelEncoder()
# y_train_lgb = le.fit_transform(y_train)

In [ ]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# scores = cross_validate(LGB_model, X_train_lgb, y_train_lgb, cv=5)
# print(np.mean(scores['test_score']))

In [ ]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_test)
# y_pred = le.inverse_transform(y_pred)

# print('accuracy :',accuracy_score(y_test, y_pred))
# print('='*60)
# print(confusion_matrix(y_test, y_pred))
# print('='*60)
# print(classification_report(y_test, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'LightGBM', 'accuracy': accuracy_score(y_test, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

### (7) 알고리즘7 : ANN

In [ ]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Dense, Dropout
# model = Sequential()

In [ ]:
# model.add(Input(shape=(561,)))
# model.add(Dense(283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=6, activation='softmax'))

In [ ]:
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# model.fit(X_train, y_train_encoded, batch_size = 32, epochs = 50)

In [ ]:
# y_pred = model.predict(X_test)
# y_pred = np.argmax(y_pred, axis=1)

In [ ]:
# y_pred = le.inverse_transform(y_pred)

# print('accuracy :',accuracy_score(y_test, y_pred))
# print('='*60)
# print(confusion_matrix(y_test, y_pred))
# print('='*60)
# print(classification_report(y_test, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'ANN', 'accuracy': accuracy_score(y_test, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

# =====특성 중요도=======================

## 방법1. 랜덤포레스트에서 추출한 feature_importance_ 사용

In [ ]:
kappa_df = pd.read_csv('모델 kappa(그리드서치).csv')
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트)
0,RandomForest,0.980076,0.981149
1,LogisticRegression,0.997722,0.997644
2,SVM,0.997721,0.992932
3,DecisionTreeClassifier,0.904857,0.896449


In [ ]:
rf_model = joblib.load('rf_model_gs.pkl')

In [ ]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,tBodyAccMag-std(),0.036021
1,tGravityAccMag-std(),0.032272
2,"tGravityAcc-arCoeff()-Z,2",0.028924
3,fBodyAccMag-std(),0.028493
4,fBodyAccMag-energy(),0.027746
...,...,...
556,"fBodyAcc-bandsEnergy()-49,64.1",0.000022
557,fBodyAcc-min()-Z,0.000020
558,"fBodyAcc-bandsEnergy()-33,48.1",0.000019
559,"fBodyAccJerk-bandsEnergy()-49,64",0.000016


In [ ]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [ ]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [ ]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 287


In [ ]:
selected_cum_importance = RFmodel_fi_df.loc[:len(selected_features), 'feature_importance'].sum()
print(f"상위 {len(selected_features)+1}개 변수의 누적 중요도: {selected_cum_importance:.4f}")

상위 288개 변수의 누적 중요도: 0.9502


In [ ]:
X_train_fi = train_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]
y_train_fi = train_data.loc[:, 'Activity']

In [ ]:
X_test_fi = test_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]

### (1) 랜덤 포레스트

In [ ]:
# feature 개수가 줄었으므로(변경됬으므로) 그리드서치 다시 해보기

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [ ]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 90}
0.9812150899688208


In [ ]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9890282131661442
[[225   0   3]
 [  0 193   2]
 [  1   1 213]]
                    precision    recall  f1-score   support

           WALKING       1.00      0.99      0.99       228
WALKING_DOWNSTAIRS       0.99      0.99      0.99       195
  WALKING_UPSTAIRS       0.98      0.99      0.98       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9835


In [ ]:
joblib.dump(rf_model, 'rf_model_gs_fi.pkl')

['rf_model_gs_fi.pkl']

In [ ]:
kappa_df.loc[0, 'gs_kappa(fi_교차검증)'] = gs.best_score_
kappa_df.loc[0, 'gs_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509
1,LogisticRegression,0.997722,0.997644,NaN,NaN
2,SVM,0.997721,0.992932,NaN,NaN
3,DecisionTreeClassifier,0.904857,0.896449,NaN,NaN


### (2) LinearRegression

In [ ]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [ ]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}],
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
0.9982907148771363


In [ ]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
joblib.dump(lr_model, 'lr_model_gs_fi.pkl')

['lr_model_gs_fi.pkl']

In [ ]:
kappa_df.loc[1, 'gs_kappa(fi_교차검증)'] = gs.best_score_
kappa_df.loc[1, 'gs_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644
2,SVM,0.997721,0.992932,NaN,NaN
3,DecisionTreeClassifier,0.904857,0.896449,NaN,NaN


### (3) SVM

In [ ]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [ ]:
gs = GridSearchCV(SVC(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 10, 'kernel': 'rbf'}
1.0


In [ ]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
joblib.dump(svm_model, 'svm_model_gs_fi.pkl')

['svm_model_gs_fi.pkl']

In [ ]:
kappa_df.loc[2, 'gs_kappa(fi_교차검증)'] = gs.best_score_
kappa_df.loc[2, 'gs_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644
3,DecisionTreeClassifier,0.904857,0.896449,NaN,NaN


### (4) DecisionTreeClassifier

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_fi, y_train_fi)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8}
0.9065863286780903


In [ ]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.945141065830721
[[215   5   8]
 [  2 187   6]
 [  5   9 201]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.94      0.96       228
WALKING_DOWNSTAIRS       0.93      0.96      0.94       195
  WALKING_UPSTAIRS       0.93      0.93      0.93       215

          accuracy                           0.95       638
         macro avg       0.94      0.95      0.94       638
      weighted avg       0.95      0.95      0.95       638

Cohen's Kappa Score: 0.9176


In [ ]:
joblib.dump(dt_model, 'dt_model_gs_fi.pkl')

['dt_model_gs_fi.pkl']

In [ ]:
kappa_df.loc[3, 'gs_kappa(fi_교차검증)'] = gs.best_score_
kappa_df.loc[3, 'gs_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604


In [ ]:
kappa_df.to_csv('모델 kappa(그리드서치_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [ ]:
kappa_df = pd.read_csv('모델 kappa(그리드서치_fi).csv')
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 100
누적 설명력: 0.9504960804927014


In [ ]:
X_test_pca = pca.transform(X_test)

### 랜덤포레스트

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [ ]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 100}
0.9378606246379804


In [ ]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9623824451410659
[[227   0   1]
 [  4 186   5]
 [ 11   3 201]]
                    precision    recall  f1-score   support

           WALKING       0.94      1.00      0.97       228
WALKING_DOWNSTAIRS       0.98      0.95      0.97       195
  WALKING_UPSTAIRS       0.97      0.93      0.95       215

          accuracy                           0.96       638
         macro avg       0.96      0.96      0.96       638
      weighted avg       0.96      0.96      0.96       638

Cohen's Kappa Score: 0.9434


In [ ]:
kappa_df.loc[0, 'gs_kappa(pca_교차검증)'] = gs.best_score_
kappa_df.loc[0, 'gs_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,NaN,NaN
2,SVM,0.997721,0.992932,1.000000,0.997644,NaN,NaN
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,NaN,NaN


### 로지스틱회귀

In [ ]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [ ]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}],
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
0.9937349248691764


In [ ]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
kappa_df.loc[1, 'gs_kappa(pca_교차검증)'] = gs.best_score_
kappa_df.loc[1, 'gs_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644,NaN,NaN
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,NaN,NaN


### SVM

In [ ]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [ ]:
gs = GridSearchCV(SVC(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 1, 'kernel': 'rbf'}
0.9977229316219093


In [ ]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 1.0
[[228   0   0]
 [  0 195   0]
 [  0   0 215]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 1.0000


In [ ]:
kappa_df.loc[2, 'gs_kappa(pca_교차검증)'] = gs.best_score_
kappa_df.loc[2, 'gs_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,NaN,NaN


### 결정트리

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_pca, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8}
0.8178329444230739


In [ ]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.8761755485893417
[[214   7   7]
 [ 18 161  16]
 [ 17  14 184]]
                    precision    recall  f1-score   support

           WALKING       0.86      0.94      0.90       228
WALKING_DOWNSTAIRS       0.88      0.83      0.85       195
  WALKING_UPSTAIRS       0.89      0.86      0.87       215

          accuracy                           0.88       638
         macro avg       0.88      0.87      0.87       638
      weighted avg       0.88      0.88      0.88       638

Cohen's Kappa Score: 0.8135


In [ ]:
kappa_df.loc[3, 'gs_kappa(pca_교차검증)'] = gs.best_score_
kappa_df.loc[3, 'gs_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,0.817833,0.813517


In [ ]:
kappa_df.to_csv('모델 kappa(그리드서치_fi_pca).csv', index=False, encoding='utf-8-sig')

## 방법3: permutation_importance()

In [ ]:
kappa_df = pd.read_csv('모델 kappa(그리드서치_fi_pca).csv')
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,0.817833,0.813517


In [35]:
rf_model = joblib.load('rf_model_gs.pkl')
lr_model = joblib.load('lr_model_gs.pkl')
svm_model = joblib.load('svm_model_gs.pkl')
dt_model = joblib.load('dt_model_gs.pkl')

### 랜덤포레스트

In [ ]:
model = rf_model

# permutation importance 계산
result_rf = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result_rf.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 34


In [ ]:
importances_df.to_csv('rf_permutation_importance_DataFrame(그리드서치).csv', index=False, encoding='utf-8-sig')

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'n_estimators': [90, 100, 120],
          'max_depth': [4, 6, 8]
          }

In [ ]:
gs = GridSearchCV(RandomForestClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_sel, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [90, 100, 120]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8, 'n_estimators': 100}
0.9800821554071458


In [ ]:
rf_model = gs.best_estimator_
y_pred = rf_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9827586206896551
[[223   0   5]
 [  2 190   3]
 [  0   1 214]]
                    precision    recall  f1-score   support

           WALKING       0.99      0.98      0.98       228
WALKING_DOWNSTAIRS       0.99      0.97      0.98       195
  WALKING_UPSTAIRS       0.96      1.00      0.98       215

          accuracy                           0.98       638
         macro avg       0.98      0.98      0.98       638
      weighted avg       0.98      0.98      0.98       638

Cohen's Kappa Score: 0.9741


In [ ]:
kappa_df.loc[0, 'gs_kappa(pi_교차검증)'] = gs.best_score_
kappa_df.loc[0, 'gs_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트),gs_kappa(pi_교차검증),gs_kappa(pi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395,0.980082,0.974081
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644,NaN,NaN
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000,NaN,NaN
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,0.817833,0.813517,NaN,NaN


### 로지스틱회귀

In [36]:
model = lr_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)

In [37]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 7


In [38]:
importances_df.to_csv('lr_permutation_importance_DataFrame(그리드서치).csv', index=False, encoding='utf-8-sig')

In [39]:
params= [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]},
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 0.5, 1, 2, 5, 10, 100]}
]

In [40]:
gs = GridSearchCV(LogisticRegression(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_sel, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42), n_jobs=-1,
             param_grid=[{'C': [0.1, 0.5, 1, 2, 5, 10, 100],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.1, 0.5, 1, 2, 5, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}],
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [41]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
0.6706894560563489


In [42]:
lr_model = gs.best_estimator_
y_pred = lr_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.799373040752351
[[176   2  50]
 [  5 178  12]
 [ 48  11 156]]
                    precision    recall  f1-score   support

           WALKING       0.77      0.77      0.77       228
WALKING_DOWNSTAIRS       0.93      0.91      0.92       195
  WALKING_UPSTAIRS       0.72      0.73      0.72       215

          accuracy                           0.80       638
         macro avg       0.81      0.80      0.80       638
      weighted avg       0.80      0.80      0.80       638

Cohen's Kappa Score: 0.6983


In [ ]:
kappa_df.loc[1, 'gs_kappa(pi_교차검증)'] = gs.best_score_
kappa_df.loc[1, 'gs_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트),gs_kappa(pi_교차검증),gs_kappa(pi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395,0.980082,0.974081
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644,0.670689,0.698342
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000,NaN,NaN
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,0.817833,0.813517,NaN,NaN


### SVM

In [ ]:
# permutation importance 계산
result = result_rf
# 선택된 피처수가 0이 나옴. 그래서 그냥 랜덤포레스트에서 추출한 변수를 사용하겠음.

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 34


In [ ]:
importances_df.to_csv('svm_permutation_importance_DataFrame(그리드서치).csv', index=False, encoding='utf-8-sig')

In [ ]:
params = {'kernel': ['linear', 'rbf', 'sigmoid'],
          'C':  [0.01, 0.1, 1, 10, 100, 1000],
          }

In [ ]:
gs = GridSearchCV(SVC(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_sel, y_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'C': 100, 'kernel': 'rbf'}
0.9914557089372483


In [ ]:
svm_model = gs.best_estimator_
y_pred = svm_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9858934169278997
[[226   0   2]
 [  1 193   1]
 [  5   0 210]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.99      0.98       228
WALKING_DOWNSTAIRS       1.00      0.99      0.99       195
  WALKING_UPSTAIRS       0.99      0.98      0.98       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9788


In [ ]:
kappa_df.loc[2, 'gs_kappa(pi_교차검증)'] = gs.best_score_
kappa_df.loc[2, 'gs_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트),gs_kappa(pi_교차검증),gs_kappa(pi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395,0.980082,0.974081
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644,0.670689,0.698342
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000,0.991456,0.978790
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,0.817833,0.813517,NaN,NaN


### 결정 트리

In [ ]:
model = dt_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 41


In [ ]:
importances_df.to_csv('dt_permutation_importance_DataFrame(그리드서치).csv', index=False, encoding='utf-8-sig')

In [ ]:
params = {'criterion': ['gini', 'entropy'],
          'max_depth': [2, 4, 6, 8, 10],
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, scoring=kappa_scorer, cv=5, n_jobs=-1)
gs.fit(X_train_sel, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, 10]},
             scoring=make_scorer(cohen_kappa_score, response_method='predict'))

In [ ]:
print(gs.best_params_)
print(gs.best_score_)

{'criterion': 'entropy', 'max_depth': 8}
0.9316834536870612


In [ ]:
dt_model = gs.best_estimator_
y_pred = dt_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9357366771159875
[[210   5  13]
 [  2 187   6]
 [  8   7 200]]
                    precision    recall  f1-score   support

           WALKING       0.95      0.92      0.94       228
WALKING_DOWNSTAIRS       0.94      0.96      0.95       195
  WALKING_UPSTAIRS       0.91      0.93      0.92       215

          accuracy                           0.94       638
         macro avg       0.94      0.94      0.94       638
      weighted avg       0.94      0.94      0.94       638

Cohen's Kappa Score: 0.9035


In [ ]:
kappa_df.loc[3, 'gs_kappa(pi_교차검증)'] = gs.best_score_
kappa_df.loc[3, 'gs_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
kappa_df

,모델,gs_kappa(교차검증),gs_kappa(테스트),gs_kappa(fi_교차검증),gs_kappa(fi_테스트),gs_kappa(pca_교차검증),gs_kappa(pca_테스트),gs_kappa(pi_교차검증),gs_kappa(pi_테스트)
0,RandomForest,0.980076,0.981149,0.981215,0.983509,0.937861,0.943395,0.980082,0.974081
1,LogisticRegression,0.997722,0.997644,0.998291,0.997644,0.993735,0.997644,0.670689,0.698342
2,SVM,0.997721,0.992932,1.000000,0.997644,0.997723,1.000000,0.991456,0.978790
3,DecisionTreeClassifier,0.904857,0.896449,0.906586,0.917604,0.817833,0.813517,0.931683,0.903474


In [ ]:
kappa_df.to_csv('모델 kappa(그리드서치_fi_pca_pi).csv', index=False, encoding='utf-8-sig')